**NLP 기초: CountVectorizer를 이용한
뉴스 그룹 대화 분류(코사인유사성)**



In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']
newsgroups = fetch_20newsgroups(subset='train', categories=categories,
remove=('headers', 'footers', 'quotes'))

In [ ]:
data, labels = [], []
for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:20]
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])

In [ ]:
print(f"### 데이터 로드 결과 ###")
print(f"추출된 총 문서 수: {len(data)}개")
print(f"선택된 카테고리: {categories}")
print("-" * 40)

### 데이터 로드 결과 ###
추출된 총 문서 수: 60개
선택된 카테고리: ['comp.graphics', 'sci.space', 'talk.religion.misc']
----------------------------------------


In [ ]:
vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)

In [ ]:
print(f"### 벡터화 결과 ###")
print(f"학습된 단어 사전 크기: {len(vectorizer.get_feature_names_out())}개")
print(f"행렬 크기 (문서, 단어): {count_matrix.shape}")
print("-" * 40)

### 벡터화 결과 ###
학습된 단어 사전 크기: 2342개
행렬 크기 (문서, 단어): (60, 2342)
----------------------------------------


In [ ]:
def classify_text(input_text):
    input_vec = vectorizer.transform([input_text])
    sim = cosine_similarity(input_vec, count_matrix)
    best_idx = np.argmax(sim)
    return labels[best_idx], sim[0][best_idx]

In [ ]:
test_sentences = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god."
]
for s in test_sentences:
    cat, score = classify_text(s)
    print(f"문장: {s[:30]}... | 예측: {cat} | 유사도: {score:.4f}")

문장: The rocket launched into orbit... | 예측: sci.space | 유사도: 0.0870
문장: A new 3D rendering technique f... | 예측: comp.graphics | 유사도: 0.1952
문장: Theological discussions on fai... | 예측: talk.religion.misc | 유사도: 0.3430




> **고찰 과제 (보고서 포함 내용)**

>> Q1.유사도가 0.0000이 나오는 이유 분석 특정 문장(예: "Exploring the mars with a robotic
rover.")을 입력했을 때 유사도가 0이 나오거나 엉뚱한 카테고리가 매칭되는 이유를
CountVectorizer의 동작 원리와 관련지어 설명하세요.
>>> CountVectorizer는 학습 데이터에 포함된 단어들로 사전을 만듭니다. 만약 사용자가 입력한 문장에 포함된 단어(예: 'rover', 'mars')가 학습용으로 사용된 60개의 문서에 단 한 번도 등장하지 않았다면, 해당 문장은 모든 값이 0인 벡터가 됩니다.주제별로 단 20개의 샘플만 사용했기 때문에 어휘 사전이 매우 제한적입니다.분자의 내적(Dot Product) 값이 0이 되어 코사인 유사도가 0으로 계산되며, 이 경우 시스템은 가장 처음에 있는 카테고리나 무작위 카테고리를 매칭하게 됩니다.



>>Q2. 성능 개선 실험 유사도 점수를 높이고 분류 정확도를 올리기 위해 다음 중 하나를 시도하고
결과를 비교하세요.
>*   주제별 샘플 수를 20개에서 100개로 늘렸을 때의 변화.
>*   CountVectorizer 대신 TfidfVectorizer를 사용했을 때의 차이점.
>*   ngram_range=(1, 2) 설정을 추가했을 때의 영향.

>>>>어휘 사전의 확장: 샘플 수를 100개로 늘리면 CountVectorizer가 학습할 수 있는 단어의 종류가 훨씬 다양해집니다. 이로 인해 "rover"나 "mars"와 같이 특정 주제에 핵심적인 단어들이 사전에 포함될 확률이 높아집니다.유사도 0.0000 문제 해결: 샘플 데이터가 적을 때는 입력 문장의 단어가 사전에 없어 유사도가 0이 나오는 경우가 많았으나, 데이터가 늘어남에 따라 입력 문장과 기존 문서 간의 공통 단어가 존재할 가능성이 커져 유사도 점수가 0보다 높게 측정됩니다.분류 정확도 향상: 더 많은 문맥 정보를 확보함으로써, 새로운 문장이 들어왔을 때 해당 주제(예: sci.space)와의 코사인 유사도가 다른 주제보다 명확하게 높게 나타나 분류 정확도가 개선됩니다.

>>>>단순히 데이터의 양을 늘리는 것만으로도 빈도 기반 벡터화 모델의 고질적인 문제인 '희소성(Sparsity)' 문제를 어느 정도 완화할 수 있으며, 이는 모델의 신뢰도 상승으로 이어집니다.





